In [ ]:
# --- Cell 0: Environment check ---
import os
# Tell PyTorch to use expandable memory segments — prevents fragmentation OOM
# (must be set before any CUDA operation is performed)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import subprocess, sys, torch

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} ({props.total_memory / 1024**3:.1f} GB VRAM)")
    torch.backends.cudnn.benchmark = False  # saves ~2 GB cuDNN workspace
else:
    print("WARNING: No GPU detected. Generation and training will be slow.")
print(f"Device: {device}")

import random, numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("Seeds set.")

In [ ]:
# --- Cell 1: Imports and config ---
import sys
from pathlib import Path

# Resolve project root: works on Colab (/content/sr_v3) and locally
if Path("/content/sr_v3").exists():
    PROJECT_ROOT = Path("/content/sr_v3")
else:
    # Local: script lives inside sr_v3/ or a parent directory
    for _candidate in [Path.cwd(), Path.cwd().parent]:
        if (_candidate / "data_v3" / "sr" / "config.py").exists():
            PROJECT_ROOT = _candidate
            break
    else:
        PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data_v3"
_sr_pkg  = DATA_DIR   # sr/ package lives at data_v3/sr/

if str(_sr_pkg) not in sys.path:
    sys.path.insert(0, str(_sr_pkg))

if not (_sr_pkg / "sr" / "config.py").exists():
    raise RuntimeError(
        f"sr package not found at {_sr_pkg}.\n"
        "Colab: ensure the repo is cloned to /content/sr_v3.\n"
        "Local: run this notebook from the sr_v3/ directory."
    )

from sr.config import SRConfig

cfg = SRConfig()

print("=== GeneratorConfig ===")
print(f"  image: {cfg.generator.image_width}x{cfg.generator.image_height}")
print(f"  candles: {cfg.generator.num_candles_min}-{cfg.generator.num_candles_max}")
print(f"  zone_magnet_strength: {cfg.generator.zone_magnet_strength}")
print(f"  dark_theme_probability: {cfg.generator.dark_theme_probability}")
print()
print("=== ScenarioWeights ===")
for name, w in zip(cfg.scenarios.names(), cfg.scenarios.weights()):
    print(f"  {name}: {w:.2f}")
print(f"  sum: {sum(cfg.scenarios.weights()):.4f}")
print()
print("=== DatasetConfig ===")
print(f"  num_examples: {cfg.dataset.num_examples}")
print(f"  split: {cfg.dataset.train_frac}/{cfg.dataset.val_frac}/{cfg.dataset.test_frac}")
print()
print("=== ModelConfig ===")
print(f"  base_channels: {cfg.model.base_channels}")
print(f"  out_channels: {cfg.model.out_channels}")
print()
print("=== TrainingConfig ===")
print(f"  epochs: {cfg.training.num_epochs}, batch: {cfg.training.batch_size}")
print(f"  lr: {cfg.training.lr_initial:.2e}, wd: {cfg.training.weight_decay:.2e}")
print(f"  pos_weights: {cfg.training.channel_pos_weights()}")
print()
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_DIR     : {DATA_DIR}")


In [ ]:
# --- Cell 2: GPU generation + JPEG rendering ---
# Generates all OHLC data on GPU, renders candlestick charts as JPEG (no matplotlib),
# saves OHLCs as CSVs, and writes labels_v3.jsonl.
import json, math, time, concurrent.futures, random as _rand
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm

from sr.data.generator import _generate_batch_gpu, _example_to_record

images_dir = DATA_DIR / "images"
ohlc_dir   = DATA_DIR / "ohlc"
images_dir.mkdir(parents=True, exist_ok=True)
ohlc_dir.mkdir(parents=True, exist_ok=True)


# ── GPU candlestick renderer ──────────────────────────────────────────────────

def render_batch_gpu(ohlc_list, lengths, price_mins, price_maxs,
                     H, W, dark_themes, draw_grids, device):
    """Render a mini-batch of OHLC charts entirely on GPU.

    Returns [N, 3, H, W] float32 tensor (values in [0, 1]).
    """
    N     = len(ohlc_list)
    max_T = max(lengths)

    pmins = torch.tensor(price_mins, dtype=torch.float32, device=device)  # [N]
    pmaxs = torch.tensor(price_maxs, dtype=torch.float32, device=device)

    # Initialise canvas with per-example background colour
    bg = torch.tensor([0.08 if d else 0.95 for d in dark_themes],
                      dtype=torch.float32, device=device)
    canvas = bg.view(N, 1, 1, 1).expand(N, 3, H, W).clone()

    # Pack OHLC into [N, max_T, 4] (padded with zeros beyond each example's length)
    ohlc_t = torch.zeros(N, max_T, 4, dtype=torch.float32, device=device)
    for i, arr in enumerate(ohlc_list):
        T = lengths[i]
        ohlc_t[i, :T] = torch.from_numpy(arr[:T]).to(device)

    def p2y(prices_n):
        """[N] prices → [N] int64 pixel row indices (y=0 is top = highest price)."""
        pct = (pmaxs - prices_n) / (pmaxs - pmins + 1e-8)
        return (pct * (H - 1)).clamp(0, H - 1).long()

    margin   = max(2, W // 50)
    cw       = max(2, (W - 2 * margin) // max_T)  # candle column width

    # Bull / bear colours per example [N, 3]
    dark_t = torch.tensor(dark_themes, device=device)  # [N] bool
    bull = torch.where(
        dark_t.view(N, 1),
        torch.tensor([[0.20, 0.75, 0.40]], device=device).expand(N, 3),
        torch.tensor([[0.10, 0.65, 0.30]], device=device).expand(N, 3),
    )
    bear = torch.where(
        dark_t.view(N, 1),
        torch.tensor([[0.90, 0.20, 0.20]], device=device).expand(N, 3),
        torch.tensor([[0.80, 0.10, 0.15]], device=device).expand(N, 3),
    )

    for t in range(max_T):
        valid = torch.tensor([t < l for l in lengths], device=device)
        if not valid.any():
            break

        xc = int(margin + t * cw + cw // 2)
        x0 = max(0, xc - cw // 2)
        x1 = min(W, x0 + max(1, cw - 1))

        o_px = p2y(ohlc_t[:, t, 0])   # open  [N]
        h_px = p2y(ohlc_t[:, t, 1])   # high
        l_px = p2y(ohlc_t[:, t, 2])   # low
        c_px = p2y(ohlc_t[:, t, 3])   # close

        bt      = torch.minimum(o_px, c_px)   # body top row
        bb      = torch.maximum(o_px, c_px)   # body bottom row
        bullish = (ohlc_t[:, t, 3] >= ohlc_t[:, t, 0])

        for i in range(N):
            if not valid[i]:
                continue

            col = bull[i] if bullish[i] else bear[i]   # [3]

            # wick (single-pixel wide at candle centre)
            wr = slice(int(h_px[i].item()), min(int(l_px[i].item()) + 1, H))
            canvas[i, :, wr, min(xc, W - 1)] = (col * 0.75).unsqueeze(1)

            # body
            br = slice(int(bt[i].item()), min(int(bb[i].item()) + 1, H))
            if br.start < br.stop:
                canvas[i, :, br, x0:x1] = col.view(3, 1, 1)

    return canvas  # [N, 3, H, W]


def save_images_fast(imgs_cpu, paths, quality=85, n_threads=16):
    """Save [N, 3, H, W] float32 tensor as JPEG files using a thread pool."""
    def _one(args):
        arr, p = args
        np_img = (arr.permute(1, 2, 0).numpy() * 255).clip(0, 255).astype(np.uint8)
        Image.fromarray(np_img).save(p, format="JPEG", quality=quality)

    with concurrent.futures.ThreadPoolExecutor(max_workers=n_threads) as ex:
        list(ex.map(_one, zip(imgs_cpu, paths)))


# ── Main generation loop ──────────────────────────────────────────────────────

GEN_BATCH    = 512   # examples per OHLC generation batch
RENDER_BATCH = 64    # examples per GPU render sub-batch

rng = torch.Generator(device=device)
rng.seed()

all_records = []
total       = cfg.dataset.num_examples
t0          = time.time()

for bi in tqdm(range(math.ceil(total / GEN_BATCH)), desc="Generating"):
    actual   = min(GEN_BATCH, total - bi * GEN_BATCH)
    examples = _generate_batch_gpu(actual, cfg, device, rng)

    # Render in sub-batches to limit peak VRAM usage
    for rb0 in range(0, len(examples), RENDER_BATCH):
        rb      = examples[rb0 : rb0 + RENDER_BATCH]
        dark    = [_rand.random() < cfg.generator.dark_theme_probability for _ in rb]
        grids   = [_rand.random() < cfg.generator.grid_probability       for _ in rb]
        imgs_g  = render_batch_gpu(
            [e.ohlc          for e in rb],
            [e.num_candles   for e in rb],
            [e.price_range[0] for e in rb],
            [e.price_range[1] for e in rb],
            cfg.generator.image_height,
            cfg.generator.image_width,
            dark, grids, device,
        )
        save_images_fast(
            imgs_g.cpu(),
            [str(images_dir / f"{e.example_id}.jpg") for e in rb],
        )

    # Save OHLCs as CSVs (fast – pure numpy)
    for e in examples:
        pd.DataFrame(e.ohlc, columns=["open", "high", "low", "close"]).to_csv(
            ohlc_dir / f"{e.example_id}.csv", index=False
        )

    all_records.extend(_example_to_record(e) for e in examples)

# Write labels JSONL
labels_path = DATA_DIR / "labels_v3.jsonl"
with open(labels_path, "w") as f:
    for rec in all_records:
        f.write(json.dumps(rec) + "\n")

elapsed = time.time() - t0
print(f"\nDone: {len(all_records)} examples in {elapsed / 60:.1f} min")
print(f"JPEGs : {len(list(images_dir.glob('*.jpg')))} / {total}")
print(f"OHLCs : {len(list(ohlc_dir.glob('*.csv')))} / {total}")
print(f"Labels: {labels_path}")

from collections import Counter
sc = Counter(r["scenario"] for r in all_records)
print("\nScenario distribution:")
for s, c in sorted(sc.items()):
    print(f"  {s}: {c} ({c / total * 100:.1f}%)")


In [ ]:
# --- Cell 3: Preview batch (one chart per scenario) ---
import json, random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
import numpy as np
from sr.config import ZoneLabel, ZoneRole
from sr.data.renderer import render_chart_with_zones
from sr.data.labels import price_to_pixel_y

with open(DATA_DIR / "labels_v3.jsonl") as f:
    records = [json.loads(line) for line in f]

def zone_from_dict(d):
    return ZoneLabel(
        zone_id=d["zone_id"], role=ZoneRole(d["role"]),
        low_price=d["low_price"], high_price=d["high_price"],
        center_price=d["center_price"], touch_count=d["touch_count"],
        strength=d["strength"], is_active=d["is_active"],
        center_y=d["center_y"], height_px=d["height_px"],
        confluence_bonus=d.get("confluence_bonus", 0.0),
    )

# One record per scenario
scenario_to_record = {}
for r in records:
    if r["scenario"] not in scenario_to_record:
        scenario_to_record[r["scenario"]] = r

preview_records = list(scenario_to_record.values())[:11]

fig, axes = plt.subplots(len(preview_records), 2, figsize=(16, 4 * len(preview_records)))

for row, rec in enumerate(preview_records):
    # Left: saved JPEG chart
    img_path = DATA_DIR / "images" / f"{rec['id']}.jpg"
    axes[row, 0].imshow(mpimg.imread(str(img_path)))
    axes[row, 0].set_title(f"{rec['scenario']} | {rec['id']}")
    axes[row, 0].axis("off")

    # Right: matplotlib render with zone overlays
    ohlc_csv = DATA_DIR / "ohlc" / f"{rec['id']}.csv"
    if ohlc_csv.exists():
        ohlc = pd.read_csv(ohlc_csv).values.astype("float32")
        zones = [zone_from_dict(z) for z in rec["zones"]]
        tmp_path = f"/tmp/preview_{rec['id']}.png"
        render_chart_with_zones(ohlc, zones, cfg.generator, tmp_path)
        axes[row, 1].imshow(mpimg.imread(tmp_path))
        axes[row, 1].set_title(f"Zones: {len(zones)}")
    else:
        axes[row, 1].set_title("OHLC CSV not found — run Cell 2 first")
    axes[row, 1].axis("off")

plt.tight_layout()
plt.show()
print(f"Preview complete. {len(preview_records)} scenarios shown.")


In [ ]:
# --- Cell 4: Build dataset and dataloaders ---
import json
from collections import Counter
from sr.data.dataset import build_dataloaders

train_loader, val_loader, test_loader = build_dataloaders(DATA_DIR, cfg, device)

# Verify a single batch
batch = next(iter(train_loader))
print(f"Image batch shape : {batch['image'].shape}")
print(f"Target batch shape: {batch['target'].shape}")
print(f"Scenarios (batch) : {batch['scenario']}")
print(f"Num zones (batch) : {batch['num_zones']}")

# Scenario distribution from labels file (fast — no full loader iteration)
with open(DATA_DIR / "labels_v3.jsonl") as f:
    _recs = [json.loads(l) for l in f]
sc = Counter(r["scenario"] for r in _recs)
print(f"\nFull dataset ({len(_recs)} examples) scenario distribution:")
for s, c in sorted(sc.items()):
    print(f"  {s}: {c} ({c / len(_recs) * 100:.1f}%)")


In [ ]:
# --- Cell 5: Instantiate model and verify ---
import torch
import matplotlib.pyplot as plt
from sr.model.net import SupportResistanceHeatmapNetV3
from sr.model.loss import SoftHeatmapLossV3

model = SupportResistanceHeatmapNetV3(dropout=cfg.model.dropout).to(device)
print(f"Model parameters: {model.count_parameters():,}")

torch.cuda.reset_peak_memory_stats()

# autocast keeps block1 activations in float16 (~2 GB vs ~9 GB in float32)
batch   = next(iter(val_loader))
images  = batch["image"].to(device)
targets = batch["target"].to(device)

_autocast = torch.amp.autocast("cuda" if device.type == "cuda" else "cpu")
with torch.no_grad(), _autocast:
    logits = model(images)
    preds  = torch.sigmoid(logits)

print(f"Input shape:  {images.shape}")
print(f"Output shape: {logits.shape}")
assert logits.shape == (images.shape[0], 5, cfg.model.output_height), \
    f"Unexpected output shape: {logits.shape}"
print("Shape assertion passed.")
print(f"Peak VRAM: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

loss_fn   = SoftHeatmapLossV3(cfg.training).to(device)
loss_dict = loss_fn(logits.float(), targets)
print(f"Loss={loss_dict['loss'].item():.4f} | "
      f"BCE={loss_dict['bce'].item():.4f} | "
      f"PeakMSE={loss_dict['peak_mse'].item():.4f}")

channel_names = ["Support", "Resistance", "Active", "Historical", "Proximity"]
fig, axes = plt.subplots(2, 5, figsize=(18, 5))
for c, name in enumerate(channel_names):
    axes[0, c].plot(targets[0, c].cpu().numpy())
    axes[0, c].set_title(f"GT: {name}")
    axes[0, c].set_ylim(0, 1)
    axes[1, c].plot(preds[0, c].float().cpu().numpy())
    axes[1, c].set_title(f"Pred (untrained): {name}")
    axes[1, c].set_ylim(0, 1)
plt.tight_layout()
plt.show()


In [ ]:
# --- Cell 6: Train ---
import pandas as pd
import matplotlib.pyplot as plt
from sr.model.net import SupportResistanceHeatmapNetV3
from sr.train import loop

# Fresh model for training
model = SupportResistanceHeatmapNetV3(dropout=cfg.model.dropout).to(device)
print(f"Model parameters: {model.count_parameters():,}")

loop.run(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    cfg=cfg,
    device=device,
    output_dir=DATA_DIR,
)

# Plot training curves
hist_df = pd.read_csv(DATA_DIR / "training_history_v3.csv")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(hist_df["epoch"], hist_df["train_loss"], label="train")
axes[0].plot(hist_df["epoch"], hist_df["val_loss"],   label="val")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(hist_df["epoch"], hist_df["train_recall"], label="train")
axes[1].plot(hist_df["epoch"], hist_df["val_recall"],   label="val")
axes[1].set_title("Zone Recall @ 10px")
axes[1].legend()

axes[2].plot(hist_df["epoch"], hist_df["lr"])
axes[2].set_title("Learning Rate")
axes[2].set_yscale("log")

plt.tight_layout()
plt.show()


In [ ]:
# --- Cell 7: Evaluate on test set ---
import torch
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from sr.model.net import SupportResistanceHeatmapNetV3
from sr.train.loop import eval_one_epoch
from sr.model.loss import SoftHeatmapLossV3

ckpt_path = DATA_DIR / "checkpoints" / "best.pt"

ckpt  = torch.load(ckpt_path, map_location=device)
model = SupportResistanceHeatmapNetV3(dropout=cfg.model.dropout).to(device)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded epoch {ckpt['epoch']} (val_loss={ckpt['val_loss']:.4f})")

loss_fn = SoftHeatmapLossV3(cfg.training).to(device)

test_metrics = eval_one_epoch(
    model, test_loader, loss_fn, device,
    epoch=ckpt["epoch"],
    save_worst_dir=DATA_DIR / "worst_recall",
)

print("\n=== Test Set Metrics ===")
print(f"  Loss              : {test_metrics['loss']:.4f}")
print(f"  Zone Recall @10px : {test_metrics['zone_recall_10px']:.4f}")
print(f"  False Peak Rate   : {test_metrics['false_peak_rate']:.4f}")

channel_names = ["Support", "Resistance", "Active", "Historical", "Proximity"]
print("\n  Per-channel Recall @10px:")
for name, r in zip(channel_names, test_metrics["per_channel_recall_10px"]):
    print(f"    {name}: {r:.4f}")

print("\n  Per-channel Peak MAE (px):")
for name, m in zip(channel_names, test_metrics["per_channel_mae_px"]):
    print(f"    {name}: {f'{m:.1f}' if m == m else 'N/A'}")

# Show worst-recall examples
worst_file = DATA_DIR / "worst_recall" / f"worst_recall_epoch_{ckpt['epoch']}.txt"
if worst_file.exists():
    worst_ids = worst_file.read_text().splitlines()[:6]
    if worst_ids:
        fig, axes = plt.subplots(1, len(worst_ids), figsize=(4 * len(worst_ids), 4))
        if len(worst_ids) == 1:
            axes = [axes]
        for ax, img_id in zip(axes, worst_ids):
            img_path = DATA_DIR / "images" / f"{img_id}.jpg"
            if img_path.exists():
                ax.imshow(mpimg.imread(str(img_path)))
            ax.set_title(img_id[:8])
            ax.axis("off")
        plt.suptitle("Worst Recall Examples")
        plt.tight_layout()
        plt.show()


In [ ]:
# --- Cell 8: Real ticker inference ---
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import tempfile
import yfinance as yf
from sr.model.net import SupportResistanceHeatmapNetV3
from sr.data.renderer import render_chart
from sr.data.labels import pixel_y_to_price
from sr.train.metrics import find_peaks_1d

# Load best checkpoint (cfg and DATA_DIR set in Cell 1)
ckpt  = torch.load(DATA_DIR / "checkpoints" / "best.pt", map_location=device)
model = SupportResistanceHeatmapNetV3(dropout=cfg.model.dropout).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Loaded epoch {ckpt['epoch']} for inference.")


def get_ticker_levels_v3(ticker: str, period: str = "6mo", sensitivity: str = "balanced"):
    """Run S/R inference on a real ticker."""
    import torchvision.transforms.functional as TF
    from PIL import Image

    df = yf.download(ticker, period=period, interval="1d", progress=False)
    if df.empty:
        print(f"No data for {ticker}")
        return

    ohlc = df[["Open", "High", "Low", "Close"]].values.astype("float32")

    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as f:
        tmp_path = f.name
    render_chart(ohlc, cfg.generator, tmp_path, dark_theme=False, draw_grid=True, draw_axes=False)

    img_tensor = TF.to_tensor(Image.open(tmp_path).convert("RGB")).unsqueeze(0).to(device)

    with torch.no_grad():
        preds = torch.sigmoid(model(img_tensor))[0]  # [5, H]

    price_min = float(ohlc[:, 2].min()) * 0.99
    price_max = float(ohlc[:, 1].max()) * 1.01
    profile   = cfg.inference.sensitivity_profiles[sensitivity]
    min_score = profile["min_score"]

    channel_names = ["Support", "Resistance", "Active", "Historical", "Proximity"]
    levels = []
    for ch in range(3):  # support, resistance, active
        for pk in find_peaks_1d(preds[ch].cpu(), threshold=min_score):
            price = pixel_y_to_price(pk, price_min, price_max, cfg.generator.image_height)
            levels.append({"channel": channel_names[ch], "price": price,
                           "score": preds[ch, pk].item(), "pixel_y": pk})

    levels.sort(key=lambda x: x["price"])
    print(f"\n{ticker} ({sensitivity}) — {len(levels)} levels:")
    print(f"{'Channel':<12} {'Price':>8} {'Score':>6}")
    print("-" * 30)
    for lv in levels:
        print(f"{lv['channel']:<12} ${lv['price']:>7.2f} {lv['score']:>6.3f}")

    # Annotated chart + heatmap
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    axes[0].imshow(mpimg.imread(tmp_path))
    for lv in levels:
        col = "green" if lv["channel"] == "Support" else "red" if lv["channel"] == "Resistance" else "yellow"
        axes[0].axhline(y=lv["pixel_y"], color=col, alpha=0.7, linewidth=1.5)
    axes[0].set_title(f"{ticker} — annotated levels")
    axes[0].axis("off")

    axes[1].plot(preds[0].cpu().numpy(), label="Support",    color="green", alpha=0.8)
    axes[1].plot(preds[1].cpu().numpy(), label="Resistance", color="red",   alpha=0.8)
    axes[1].plot(preds[2].cpu().numpy(), label="Active",     color="gold",  alpha=0.8)
    axes[1].set_title("Predicted Heatmaps")
    axes[1].legend()
    axes[1].set_ylim(0, 1)
    plt.tight_layout()
    plt.show()

    return levels


for ticker in ["AAPL", "SPY", "TSLA"]:
    get_ticker_levels_v3(ticker, sensitivity="balanced")
